In [1]:
from dataclasses import dataclass
import json

@dataclass
class Ride:
    lpep_pickup_datetime: int  # epoch milliseconds
    lpep_dropoff_datetime: int
    PULocationID: int
    DOLocationID: int
    passenger_count: float
    trip_distance: float
    tip_amount: float
    total_amount: float

def ride_from_row(row):
    return Ride(
        lpep_pickup_datetime=int(row['lpep_pickup_datetime'].timestamp() * 1000),
        lpep_dropoff_datetime=int(row['lpep_pickup_datetime'].timestamp() * 1000),
        PULocationID=int(row['PULocationID']),
        DOLocationID=int(row['DOLocationID']),
        passenger_count=float(row['passenger_count']),
        trip_distance=float(row['trip_distance']),
        tip_amount=float(row['tip_amount']),
        total_amount=float(row['total_amount']),
    )

In [2]:
def ride_deserializer(data):
    json_str = data.decode('utf-8')
    ride_dict = json.loads(json_str)
    return Ride(**ride_dict)

In [3]:
test_bytes = json.dumps({
    'lpep_pickup_datetime': 1730429702000,
    'lpep_dropoff_datetime': 1730429802000,
    'PULocationID': 188,
    'DOLocationID': 79,
    'passenger_count': 2.0,
    'trip_distance': 1.72,
    'tip_amount': 2.73,
    'total_amount': 5.72
}).encode('utf-8')

ride_deserializer(test_bytes)

Ride(lpep_pickup_datetime=1730429702000, lpep_dropoff_datetime=1730429802000, PULocationID=188, DOLocationID=79, passenger_count=2.0, trip_distance=1.72, tip_amount=2.73, total_amount=5.72)

In [4]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'green-trips'

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    value_deserializer=ride_deserializer
)

In [5]:
from datetime import datetime

print(f"Listening to {topic_name}...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.lpep_pickup_datetime / 1000)
    dropoff_dt = datetime.fromtimestamp(ride.lpep_dropoff_datetime / 1000)
    
    print(
        f"Received: "
        f"PU={ride.PULocationID}, "
        f"DO={ride.DOLocationID}, "
        f"passengers={ride.passenger_count}, "
        f"distance={ride.trip_distance:.2f} mi, "
        f"tip=${ride.tip_amount:.2f}, "
        f"total=${ride.total_amount:.2f}, "
        f"pickup={pickup_dt}, "
        f"dropoff={dropoff_dt}"
    )
    
    count += 1
    if count >= 10:
        print(f"\n... received {count} messages so far (stopping after 10 for demo)")
        break
        
consumer.close()

Listening to green-trips...
Received: PU=247, DO=69, passengers=1.0, distance=0.70 mi, tip=$1.70, total=$10.00, pickup=2025-10-01 00:21:47, dropoff=2025-10-01 00:21:47
Received: PU=247, DO=69, passengers=1.0, distance=0.70 mi, tip=$1.70, total=$10.00, pickup=2025-10-01 00:21:47, dropoff=2025-10-01 00:21:47
Received: PU=247, DO=69, passengers=1.0, distance=0.70 mi, tip=$1.70, total=$10.00, pickup=2025-10-01 00:21:47, dropoff=2025-10-01 00:21:47
Received: PU=66, DO=25, passengers=1.0, distance=1.61 mi, tip=$2.78, total=$16.68, pickup=2025-10-01 00:14:03, dropoff=2025-10-01 00:14:03
Received: PU=244, DO=244, passengers=1.0, distance=0.00 mi, tip=$2.20, total=$13.20, pickup=2025-10-01 00:16:44, dropoff=2025-10-01 00:16:44
Received: PU=95, DO=170, passengers=1.0, distance=10.37 mi, tip=$11.31, total=$67.85, pickup=2025-10-01 00:07:36, dropoff=2025-10-01 00:07:36
Received: PU=82, DO=138, passengers=1.0, distance=4.07 mi, tip=$6.82, total=$34.12, pickup=2025-09-30 21:10:29, dropoff=2025-09-30